# 第136章: 時空間モデリング総括 — Phase 6の統合と展望

## この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] Phase 6（Notebook 130-135）の技術体系を俯瞰し、各章の位置づけを説明できる
- [ ] U-Net系・DiT系・(2+1)D Conv系アーキテクチャの特徴を比較できる
- [ ] Sora・Runway・Stable Video Diffusion等の商用AIの技術的位置づけを説明できる
- [ ] 長尺動画生成・物理整合性など残された課題を理解できる
- [ ] Phase 7（世界モデル）への接続と展望を把握できる

## 前提知識

- Notebook 130（時間的注意機構の基礎）
- Notebook 131（Video Diffusion Models）
- Notebook 132（Diffusion Transformer / DiT）
- Notebook 133（(2+1)D Convolutionと時空間分離）
- Notebook 134（動画生成の制御：カメラ・物体運動）
- Notebook 135（物理整合性と評価指標）

---

| 項目 | 内容 |
|------|------|
| 推定学習時間 | 90-120分 |
| 難易度 | ★★★★☆（上級） |
| カテゴリ | 時空間モデリング総括 |
| 種類 | レビュー・統合（コード少なめ・概念整理中心） |

## 目次

1. [Phase 6で学んだ技術体系](#section1)
2. [アーキテクチャ比較表](#section2)
3. [商用動画生成AIの位置づけ](#section3)
4. [残された課題](#section4)
5. [Phase 7へのロードマップ](#section5)
6. [総合クイズ](#quiz)
7. [まとめとチェックリスト](#summary)

In [ ]:
# ============================================================
# 環境設定
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import warnings

warnings.filterwarnings('ignore')

# 日本語フォント設定
def setup_japanese_font():
    """日本語フォントを設定する"""
    japanese_fonts = [
        'Hiragino Sans', 'Hiragino Maru Gothic Pro',
        'Yu Gothic', 'MS Gothic',
        'Noto Sans CJK JP', 'IPAexGothic',
    ]
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# 再現性の確保
np.random.seed(42)

print(f'ライブラリのインポート完了')
print(f'日本語フォント: {font_used}')
print(f'このノートブックはPyTorch不要です（可視化と概念整理のみ）')

<a id="section1"></a>
## 1. Phase 6で学んだ技術体系

### Phase 6の全体像

Phase 6「時空間モデリング」では、**静止画の生成モデル**を**動画の生成モデル**へ拡張するための
技術体系を6つのノートブック（130-135）で学びました。

各ノートブックの位置づけを以下に整理します：

| Notebook | テーマ | 核心技術 | 位置づけ |
|----------|--------|----------|----------|
| **130** | 時間的注意機構の基礎 | Temporal Self-Attention, Causal Mask | **基盤** — 全ての時空間モデルの構成要素 |
| **131** | Video Diffusion Models | U-Net + Temporal層, (2+1)D Conv | **実装** — 画像拡散を動画に拡張 |
| **132** | Diffusion Transformer (DiT) | adaLN-Zero, Spacetime Patches | **発展** — U-NetからTransformerへ |
| **133** | (2+1)D Convolutionと時空間分離 | Factorized Conv, 計算効率 | **効率化** — 実用的なスケーリング |
| **134** | 動画生成の制御 | カメラ制御, 物体運動条件付け | **制御性** — 実用的な生成 |
| **135** | 物理整合性と評価 | 物理損失, FVD, 時間的一貫性指標 | **品質** — 信頼性の担保 |

この6章は「基盤 → 実装 → 発展 → 効率化 → 制御性 → 品質」という論理的な流れで構成されています。

In [ ]:
# ============================================================
# Phase 6 学習パスの全体マップ
# ============================================================

def visualize_phase6_learning_path():
    """Phase 6の学習パスとノートブック間の関係を可視化"""
    fig, ax = plt.subplots(figsize=(18, 11))
    
    # ノートブック情報
    notebooks = [
        ('130\n時間的注意機構\nの基礎',       0.12, 0.82, 'lightblue',
         'Temporal Attention\nCausal Mask\nSpatioTemporalBlock'),
        ('131\nVideo Diffusion\nModels',       0.38, 0.82, 'lightgreen',
         'U-Net + Temporal層\n(2+1)D Conv\nMoving-MNIST'),
        ('132\nDiffusion\nTransformer',        0.65, 0.82, 'lightyellow',
         'DiT / adaLN-Zero\nSpacetime Patches\nスケーリング則'),
        ('133\n(2+1)D Conv\n時空間分離',       0.25, 0.45, 'lightsalmon',
         'Factorized Conv\n計算量比較\n事前学習活用'),
        ('134\n動画生成の制御',                0.52, 0.45, 'plum',
         'カメラ制御\n物体運動条件付け\nControlNet拡張'),
        ('135\n物理整合性\nと評価',            0.78, 0.45, 'lightskyblue',
         '物理損失関数\nFVD / FID-vid\n時間的一貫性指標'),
    ]
    
    box_w, box_h = 0.20, 0.18
    
    for title, x, y, color, details in notebooks:
        # メインボックス
        rect = mpatches.FancyBboxPatch(
            (x - box_w/2, y - box_h/2), box_w, box_h,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='gray', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(x, y + 0.03, title, ha='center', va='center',
                fontsize=10, fontweight='bold')
        ax.text(x, y - 0.065, details, ha='center', va='center',
                fontsize=7.5, color='#333333', style='italic')
    
    # 接続矢印 — 主要な依存関係
    arrows = [
        # 上段の直線的流れ
        (0.22, 0.82, 0.28, 0.82, 'black'),
        (0.48, 0.82, 0.55, 0.82, 'black'),
        # 上段から下段へ
        (0.12, 0.73, 0.22, 0.54, 'steelblue'),     # 130 → 133
        (0.38, 0.73, 0.38, 0.54, 'steelblue'),     # 131 → 133
        (0.38, 0.73, 0.52, 0.54, 'steelblue'),     # 131 → 134
        (0.65, 0.73, 0.52, 0.54, 'steelblue'),     # 132 → 134
        (0.65, 0.73, 0.78, 0.54, 'steelblue'),     # 132 → 135
        (0.52, 0.36, 0.78, 0.36, 'gray'),          # 134 → 135
    ]
    
    for x1, y1, x2, y2, color in arrows:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.8, alpha=0.7))
    
    # Phase ラベル
    ax.text(0.38, 0.97, 'Phase 6: 時空間モデリング 技術体系マップ',
            ha='center', va='center', fontsize=16, fontweight='bold')
    
    # 行ラベル
    ax.text(0.02, 0.82, '基盤・\n実装・\n発展', ha='center', va='center',
            fontsize=9, fontweight='bold', color='gray')
    ax.text(0.02, 0.45, '効率化・\n制御・\n品質', ha='center', va='center',
            fontsize=9, fontweight='bold', color='gray')
    
    # Phase 7 への矢印
    rect_next = mpatches.FancyBboxPatch(
        (0.32, 0.08), 0.30, 0.12,
        boxstyle='round,pad=0.01',
        facecolor='gold', edgecolor='orange', linewidth=2.5, alpha=0.8
    )
    ax.add_patch(rect_next)
    ax.text(0.47, 0.14, 'Phase 7: 世界モデル\nJEPA / Dreamer / Genie',
            ha='center', va='center', fontsize=11, fontweight='bold')
    
    ax.annotate('', xy=(0.47, 0.20), xytext=(0.47, 0.36),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2.5))
    
    ax.set_xlim(-0.02, 0.95)
    ax.set_ylim(0.0, 1.02)
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_phase6_learning_path()

### 各ノートブックの核心コンセプト

#### Notebook 130: 時間的注意機構の基礎

Phase 6全体の**土台**となる章です。画像のSelf-Attention（ViT / Notebook 95）から出発し、
動画に必要な**時間軸方向のAttention**を導入しました。

- **Spatial Attention**: `(B*T, H*W, C)` — 同一フレーム内のパッチ間
- **Temporal Attention**: `(B*H*W, T, C)` — 同一位置の異なるフレーム間
- **Causal Mask**: 自己回帰生成で未来の情報を遮断する下三角マスク
- **SpatioTemporalBlock**: Spatial → Temporal → FFN の3段構成

重要な洞察: **reshapeだけでSpatialとTemporalを切り替えられる**。数式は同じAttentionです。

---

#### Notebook 131: Video Diffusion Models

画像拡散モデル（DDPM / Notebook 40-42）を動画に拡張する**最初の実装**です。

- **U-Net + Temporal層**: 既存の画像U-Netにtemporal convとtemporal attentionを追加
- **(2+1)D Convolution**: 空間2D + 時間1Dの分離畳み込みで効率化
- **事前学習の活用**: 画像で学習した空間処理の重みを再利用
- **Moving-MNIST**: 合成動画データでの訓練・生成パイプライン

---

#### Notebook 132: Diffusion Transformer (DiT)

U-NetからTransformerベースへの**パラダイムシフト**。Soraの技術的基盤です。

- **adaLN-Zero**: 拡散タイムステップをLayerNormのスケール・シフトで条件付け
- **Spacetime Patches**: 動画を時空間パッチに分割しトークン化
- **スケーリング則**: モデルサイズと計算量の関係、Chinchilla則の動画版

---

#### Notebook 133: (2+1)D Convolutionと時空間分離

実用的なスケーリングに必要な**計算効率の技術**です。

- **Factorized Convolution**: 3D Conv → 2D + 1D への分離と計算量削減
- **パラメータ効率**: 同等の表現力でパラメータ数を削減
- **事前学習の橋渡し**: 画像モデルの重みを動画モデルに転移

---

#### Notebook 134: 動画生成の制御

「ただ動画を生成する」から「意図通りの動画を生成する」への**制御性**の章です。

- **カメラ制御**: パン・ティルト・ズーム等のカメラワーク指定
- **物体運動条件付け**: トラジェクトリ（軌跡）やバウンディングボックスによる制御
- **ControlNet拡張**: 追加の制御信号を注入する手法の動画版

---

#### Notebook 135: 物理整合性と評価

生成した動画の**品質と信頼性**を評価するための技術です。

- **物理損失関数**: 重力、衝突、流体力学に基づく制約
- **FVD (Frechet Video Distance)**: 動画品質の定量指標
- **時間的一貫性**: フレーム間の整合性を測る指標群

In [ ]:
# ============================================================
# 各ノートブックの核心概念サマリ
# ============================================================

def print_phase6_summary():
    """Phase 6の各章を一行サマリで整理"""
    print("=" * 70)
    print("Phase 6 — 各章の核心コンセプトと一行サマリ")
    print("=" * 70)
    
    chapters = [
        ("130", "時間的注意機構の基礎",
         "reshapeだけでSpatial/Temporalを切り替える",
         "基盤"),
        ("131", "Video Diffusion Models",
         "画像U-Netにtemporal層を足すだけで動画拡散モデルになる",
         "実装"),
        ("132", "Diffusion Transformer (DiT)",
         "U-NetをTransformerに置き換え、スケーリング性能を改善",
         "発展"),
        ("133", "(2+1)D Conv と時空間分離",
         "3D処理を2D+1Dに分離して計算コストを劇的に削減",
         "効率化"),
        ("134", "動画生成の制御",
         "カメラ・物体運動の条件を注入して意図通りの動画を生成",
         "制御性"),
        ("135", "物理整合性と評価",
         "物理法則ベースの損失と定量指標で生成品質を担保",
         "品質"),
    ]
    
    for nb, title, insight, role in chapters:
        print(f"\n  [{role}] Notebook {nb}: {title}")
        print(f"    --> {insight}")
    
    print("\n" + "=" * 70)
    print("Phase 6は『基盤 -> 実装 -> 発展 -> 効率化 -> 制御性 -> 品質』")
    print("の6ステップで動画生成AI の技術体系を網羅しました。")

print_phase6_summary()

In [ ]:
# ============================================================
# Phase 6 の核心数式チートシート
# ============================================================

def print_key_formulas():
    """Phase 6で登場した核心数式を一覧表示"""
    print("=" * 70)
    print("Phase 6 核心数式チートシート")
    print("=" * 70)
    
    formulas = [
        ("Self-Attention",
         "Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V",
         "Notebook 130",
         "Spatial/Temporal共通。reshapeで対象軸を切り替え。"),
        ("因果マスク",
         "M[i,j] = 1 if j <= i, else 0  (下三角行列)",
         "Notebook 130",
         "自己回帰生成で未来を遮断。"),
        ("前方拡散",
         "q(x_t|x_0) = N(sqrt(a_bar_t) * x_0, (1-a_bar_t) * I)",
         "Notebook 131",
         "画像/動画共通の拡散過程。"),
        ("拡散損失",
         "L = E[||eps - eps_theta(x_t, t)||^2]",
         "Notebook 131",
         "ノイズ予測のMSE損失。"),
        ("adaLN-Zero",
         "y = gamma(c) * LayerNorm(x) + beta(c)",
         "Notebook 132",
         "条件cからgamma,betaを生成。alpha=0初期化。"),
        ("(2+1)D Conv",
         "Conv3D(x) = Conv1D_t(Conv2D_s(x))",
         "Notebook 133",
         "空間2D -> 時間1Dの分離。パラメータ削減。"),
        ("FVD",
         "FVD = ||mu_r-mu_g||^2 + Tr(S_r+S_g - 2*(S_r*S_g)^0.5)",
         "Notebook 135",
         "動画品質の定量指標（FIDの動画版）。"),
    ]
    
    for name, formula, notebook, note in formulas:
        print(f"\n  [{notebook}] {name}")
        print(f"    {formula}")
        print(f"    Note: {note}")
    
    print("\n" + "=" * 70)

print_key_formulas()

### 数式の統一的な見方

Phase 6の数式を俯瞰すると、以下の構造が見えてきます：

1. **Attention機構** (Notebook 130) が全ての基盤。Spatial/Temporalはreshapeだけの違い
2. **拡散過程** (Notebook 131) が生成のフレームワーク。画像と動画で数式は同一
3. **条件付け** (Notebook 132) がadaLN-Zeroで効率的に実現
4. **分離** (Notebook 133) が計算効率の鍵。3D → 2D+1D
5. **評価** (Notebook 135) がFVDで品質を定量化

これらの数式は独立ではなく、動画拡散モデルの中で有機的に組み合わされています。

<a id="section2"></a>
## 2. アーキテクチャ比較表

### 3つの主要アプローチ

Phase 6で学んだ動画生成アーキテクチャは、大きく3つのアプローチに分類できます：

| 項目 | U-Net系 Video Diffusion | Diffusion Transformer (DiT) | (2+1)D Conv 系 |
|------|------------------------|---------------------------|----------------|
| **基本構造** | Encoder-Decoder + Skip | 純粋なTransformer | 分離畳み込み |
| **時間モデリング** | Temporal層の挿入 | Spacetime Self-Attention | 時間1D Conv |
| **条件付け** | Cross-Attention / 連結 | adaLN-Zero | FiLM / 連結 |
| **パラメータ規模** | ~100M-1B | ~600M-10B+ | ~50M-500M |
| **スケーラビリティ** | 中（設計が複雑化） | 高（パッチ数で制御） | 中（CNNの限界） |
| **事前学習活用** | 画像SD重みを再利用 | 画像DiT重みを再利用 | 画像CNN重みを再利用 |
| **訓練コスト** | 中 | 高（大規模データ必要） | 低-中 |
| **生成品質** | 高（実績豊富） | 最高（最新SOTA） | 中-高 |
| **代表例** | Runway Gen-2, SVD | Sora, Latte | R(2+1)D, SlowFast |

それぞれの特性を詳しく見ていきましょう。

In [ ]:
# ============================================================
# アーキテクチャ比較: レーダーチャート
# ============================================================

def plot_architecture_radar():
    """3つのアーキテクチャを多次元で比較するレーダーチャート"""
    
    categories = [
        'パラメータ\n効率',
        '時間的\nモデリング力',
        'スケーラ\nビリティ',
        '訓練\nコスト効率',
        '生成\n品質',
        '事前学習\n活用度',
        '制御性',
    ]
    N = len(categories)
    
    # 各アーキテクチャのスコア (1-5)
    scores = {
        'U-Net系 Video Diffusion': [3, 3.5, 3, 3.5, 4, 4.5, 3.5],
        'Diffusion Transformer (DiT)': [2, 5, 5, 2, 5, 3.5, 4],
        '(2+1)D Conv系': [4.5, 3, 2.5, 4.5, 3, 4, 2.5],
    }
    
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # 閉じる
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    colors = ['steelblue', 'coral', 'seagreen']
    
    for (name, vals), color in zip(scores.items(), colors):
        vals_closed = vals + vals[:1]
        ax.plot(angles, vals_closed, 'o-', linewidth=2.5, label=name, color=color)
        ax.fill(angles, vals_closed, alpha=0.12, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_ylim(0, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=9)
    ax.set_title('動画生成アーキテクチャの多次元比較',
                 fontsize=15, fontweight='bold', pad=25)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_architecture_radar()

In [ ]:
# ============================================================
# アーキテクチャ比較: グループ化棒グラフ
# ============================================================

def plot_architecture_comparison_bars():
    """アーキテクチャの定量的比較（概算値ベース）"""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    architectures = ['U-Net系\nVideo Diffusion', 'DiT\n(Sora型)', '(2+1)D\nConv系']
    colors = ['steelblue', 'coral', 'seagreen']
    
    # (a) パラメータ数の比較（概算, 単位: 百万）
    params = [860, 3000, 300]
    ax = axes[0]
    bars = ax.bar(architectures, params, color=colors, alpha=0.8, edgecolor='gray')
    ax.set_ylabel('パラメータ数 (M)', fontsize=12)
    ax.set_title('(a) 代表的なパラメータ規模', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, params):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val}M', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(params) * 1.2)
    ax.grid(True, alpha=0.3, axis='y')
    
    # (b) 訓練コスト（概算, GPU時間/千時間）
    cost = [50, 500, 10]
    ax = axes[1]
    bars = ax.bar(architectures, cost, color=colors, alpha=0.8, edgecolor='gray')
    ax.set_ylabel('GPU時間 (千 A100時間)', fontsize=12)
    ax.set_title('(b) 概算訓練コスト', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, cost):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val}K', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(cost) * 1.2)
    ax.grid(True, alpha=0.3, axis='y')
    
    # (c) 最大動画長（概算, 秒）
    duration = [4, 60, 2]
    ax = axes[2]
    bars = ax.bar(architectures, duration, color=colors, alpha=0.8, edgecolor='gray')
    ax.set_ylabel('最大動画長 (秒)', fontsize=12)
    ax.set_title('(c) 生成可能な動画長', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, duration):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(duration) * 1.3)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('動画生成アーキテクチャの定量比較（概算値）',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("注意: 上記の数値は公開情報に基づく概算値です。")
    print("実際の値はモデル構成や訓練設定によって大きく異なります。")

plot_architecture_comparison_bars()

In [ ]:
# ============================================================
# 3つのアーキテクチャの内部構造比較図
# ============================================================

def plot_architecture_block_diagrams():
    """U-Net系、DiT系、(2+1)D Conv系の内部構造を並べて図示"""
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 12))
    
    def draw_block(ax, x, y, w, h, label, color, fontsize=9):
        rect = mpatches.FancyBboxPatch(
            (x - w/2, y - h/2), w, h,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='gray', linewidth=1.5
        )
        ax.add_patch(rect)
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold')
    
    def draw_arrow(ax, x1, y1, x2, y2, color='black'):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    
    # --- (a) U-Net系 Video Diffusion ---
    ax = axes[0]
    blocks_a = [
        (0.5, 0.95, 0.5, 0.06, 'Input (B,T,C,H,W)', 'lightyellow'),
        (0.5, 0.85, 0.5, 0.06, 'Conv2D + Temporal Attn', 'lightblue'),
        (0.5, 0.75, 0.4, 0.06, 'Downsample + Temp Attn', 'lightblue'),
        (0.5, 0.65, 0.3, 0.06, 'Downsample + Temp Attn', 'lightblue'),
        (0.5, 0.55, 0.25, 0.06, 'Bottleneck (ST-Attn)', 'lightsalmon'),
        (0.5, 0.45, 0.3, 0.06, 'Upsample + Temp Attn', 'lightgreen'),
        (0.5, 0.35, 0.4, 0.06, 'Upsample + Temp Attn', 'lightgreen'),
        (0.5, 0.25, 0.5, 0.06, 'Conv2D + Temporal Attn', 'lightgreen'),
        (0.5, 0.15, 0.5, 0.06, 'Output (B,T,C,H,W)', 'lightyellow'),
    ]
    for bx, by, bw, bh, label, color in blocks_a:
        draw_block(ax, bx, by, bw, bh, label, color, fontsize=8)
    for i in range(len(blocks_a) - 1):
        draw_arrow(ax, 0.5, blocks_a[i][1]-0.03, 0.5, blocks_a[i+1][1]+0.03)
    # Skip connections
    for enc_idx, dec_idx in [(1, 7), (2, 6), (3, 5)]:
        ey = blocks_a[enc_idx][1]
        dy = blocks_a[dec_idx][1]
        ax.annotate('', xy=(0.78, dy), xytext=(0.78, ey),
                    arrowprops=dict(arrowstyle='->', color='orange', lw=2,
                                   connectionstyle='arc3,rad=0.2'))
    ax.text(0.88, 0.60, 'Skip\nConnect', fontsize=8, color='orange', fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(0.05, 1.02)
    ax.axis('off')
    ax.set_title('(a) U-Net系 Video Diffusion', fontsize=13, fontweight='bold')
    
    # --- (b) DiT系 ---
    ax = axes[1]
    blocks_b = [
        (0.5, 0.95, 0.55, 0.06, 'Input Video', 'lightyellow'),
        (0.5, 0.84, 0.55, 0.06, 'Patchify (Spacetime Patches)', 'lightcoral'),
        (0.5, 0.73, 0.55, 0.06, 'DiT Block [adaLN + Attn + FFN]', '#D4EDDA'),
        (0.5, 0.62, 0.55, 0.06, 'DiT Block [adaLN + Attn + FFN]', '#D4EDDA'),
        (0.5, 0.51, 0.55, 0.06, 'DiT Block [adaLN + Attn + FFN]', '#D4EDDA'),
        (0.5, 0.38, 0.55, 0.06, 'Linear Decoder', 'lightblue'),
        (0.5, 0.27, 0.55, 0.06, 'Unpatchify', 'lightcoral'),
        (0.5, 0.16, 0.55, 0.06, 'Output Video', 'lightyellow'),
    ]
    for i, (bx, by, bw, bh, label, color) in enumerate(blocks_b):
        draw_block(ax, bx, by, bw, bh, label, color, fontsize=8)
        if i < len(blocks_b) - 1:
            draw_arrow(ax, 0.5, by-0.03, 0.5, blocks_b[i+1][1]+0.03)
    # Condition
    draw_block(ax, 0.88, 0.62, 0.18, 0.05, 't, class', 'gold', fontsize=8)
    draw_arrow(ax, 0.82, 0.62, 0.78, 0.62, 'orange')
    ax.text(0.88, 0.55, 'adaLN-Zero', fontsize=7, ha='center', color='orange')
    ax.set_xlim(0, 1)
    ax.set_ylim(0.05, 1.02)
    ax.axis('off')
    ax.set_title('(b) Diffusion Transformer (DiT)', fontsize=13, fontweight='bold')
    
    # --- (c) (2+1)D Conv系 ---
    ax = axes[2]
    blocks_c = [
        (0.5, 0.95, 0.55, 0.06, 'Input (B,C,T,H,W)', 'lightyellow'),
        (0.35, 0.82, 0.35, 0.06, '2D Spatial Conv', 'lightblue'),
        (0.35, 0.71, 0.35, 0.06, 'BN + ReLU', 'lavender'),
        (0.7, 0.71, 0.25, 0.06, '1D Temporal\nConv', 'lightgreen'),
        (0.5, 0.58, 0.55, 0.06, 'BN + ReLU', 'lavender'),
        (0.35, 0.45, 0.35, 0.06, '2D Spatial Conv', 'lightblue'),
        (0.35, 0.34, 0.35, 0.06, 'BN + ReLU', 'lavender'),
        (0.7, 0.34, 0.25, 0.06, '1D Temporal\nConv', 'lightgreen'),
        (0.5, 0.22, 0.55, 0.06, 'Pooling / Downsample', 'lightsalmon'),
        (0.5, 0.12, 0.55, 0.06, 'Output Features', 'lightyellow'),
    ]
    for bx, by, bw, bh, label, color in blocks_c:
        draw_block(ax, bx, by, bw, bh, label, color, fontsize=8)
    draw_arrow(ax, 0.5, 0.92, 0.35, 0.85)
    draw_arrow(ax, 0.35, 0.79, 0.35, 0.74)
    draw_arrow(ax, 0.52, 0.71, 0.58, 0.71, 'green')
    draw_arrow(ax, 0.7, 0.68, 0.5, 0.61, 'green')
    draw_arrow(ax, 0.5, 0.55, 0.35, 0.48)
    draw_arrow(ax, 0.35, 0.42, 0.35, 0.37)
    draw_arrow(ax, 0.52, 0.34, 0.58, 0.34, 'green')
    draw_arrow(ax, 0.7, 0.31, 0.5, 0.25, 'green')
    draw_arrow(ax, 0.5, 0.19, 0.5, 0.15)
    ax.set_xlim(0, 1)
    ax.set_ylim(0.05, 1.02)
    ax.axis('off')
    ax.set_title('(c) (2+1)D Conv系', fontsize=13, fontweight='bold')
    
    plt.suptitle('3つのアーキテクチャの内部構造比較',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_architecture_block_diagrams()

### 各アーキテクチャの進化的位置づけ

3つのアーキテクチャは、動画生成技術の**進化の段階**を反映しています：

```
Stage 1: (2+1)D Conv    → 動画理解 (認識タスク) から出発
  |
  | 生成モデルへの転用
  v
Stage 2: U-Net + Temporal → 画像拡散モデルの成功を動画に拡張
  |
  | スケーリングの限界を克服
  v
Stage 3: DiT            → Transformerの統一的アーキテクチャで品質とスケーリングを両立
```

この進化は、NLPでRNN → LSTM → Transformerへと移行した流れと平行しています。
最終的にTransformerが勝利する構図は、モダリティ（言語・画像・動画）を超えた
普遍的なパターンと言えるでしょう。

### アーキテクチャ選択のガイドライン

どのアーキテクチャを選ぶべきかは、目的とリソースによって異なります：

| ユースケース | 推奨アーキテクチャ | 理由 |
|-------------|------------------|------|
| 最高品質の動画生成 | **DiT** | スケーリングで品質が向上し続ける |
| 既存画像モデルの動画化 | **U-Net系** | Stable Diffusionの重みを活用できる |
| リアルタイム/低リソース | **(2+1)D Conv** | 計算効率が高い |
| 長尺動画 (>10秒) | **DiT** | Spacetime Patchesで柔軟に対応 |
| 研究プロトタイプ | **U-Net系** | 実装が豊富、デバッグが容易 |

In [ ]:
# ============================================================
# Attention計算量のスケーリング比較
# ============================================================

def plot_attention_scaling():
    """フレーム数に応じたAttention計算量の増加を可視化"""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # (a) フレーム数 vs Attention計算量
    T_range = np.arange(4, 129, 4)
    H, W = 64, 64
    N = H * W  # 空間トークン数
    
    spatial = T_range * N**2           # O(T * N^2)
    temporal = N * T_range**2          # O(N * T^2)
    factorized = spatial + temporal    # Spatial + Temporal
    full_st = (T_range * N)**2         # O((T*N)^2)
    
    ax = axes[0]
    ax.semilogy(T_range, spatial, 'b-o', markersize=3, label='Spatial O(T*N^2)', linewidth=2)
    ax.semilogy(T_range, temporal, 'r-s', markersize=3, label='Temporal O(N*T^2)', linewidth=2)
    ax.semilogy(T_range, factorized, 'g-^', markersize=3, label='Factorized (S+T)', linewidth=2)
    ax.semilogy(T_range, full_st, 'k--', markersize=3, label='Full ST O((TN)^2)', linewidth=2)
    ax.set_xlabel('フレーム数 T', fontsize=12)
    ax.set_ylabel('Attention計算量 (対数スケール)', fontsize=12)
    ax.set_title('(a) フレーム数と計算量の関係\n(H=W=64)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # (b) Factorized vs Full の比率
    ratio = factorized / full_st
    ax = axes[1]
    ax.semilogy(T_range, ratio, 'g-o', markersize=4, linewidth=2.5)
    ax.set_xlabel('フレーム数 T', fontsize=12)
    ax.set_ylabel('Factorized / Full 比率', fontsize=12)
    ax.set_title('(b) 分離Attentionの効率\n(Full STに対する比率)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=1e-3, color='red', linestyle='--', alpha=0.5)
    ax.text(80, 2e-3, '1000倍高速', fontsize=10, color='red')
    
    plt.suptitle('時空間Attentionの計算量スケーリング',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('Factorized Attention (Spatial + Temporal) は Full Spatiotemporal に比べて')
    print(f'T=16, 64x64 の場合: {factorized[3]/full_st[3]:.6f} 倍 = 約{full_st[3]/factorized[3]:.0f}倍高速')
    print(f'T=64, 64x64 の場合: {factorized[15]/full_st[15]:.6f} 倍 = 約{full_st[15]/factorized[15]:.0f}倍高速')

plot_attention_scaling()

### 計算量のスケーリングから得られる教訓

上のグラフから明らかなように：

1. **Full Spatiotemporal Attention は現実的でない**: フレーム数が増えると計算量が爆発的に増加
2. **Factorized (Spatial + Temporal) が実用的な選択**: Full STに対して1000倍以上高速
3. **Temporal計算量はフレーム数に二次的に増加**: T^2 の項が長尺動画のボトルネック

これがNotebook 133 (2+1)D Convolutionで学んだ「分離」の本質的な価値であり、
同時にSection 4「残された課題」で議論する長尺動画生成の技術的困難の根源です。

<a id="section3"></a>
## 3. 商用動画生成AIの位置づけ

### 主要な商用動画生成AI

Phase 6で学んだ技術は、実際の商用製品の中核を成しています。
各製品がどの技術をどう活用しているかを整理します。

---

#### Sora (OpenAI)

- **アーキテクチャ**: Diffusion Transformer (DiT)
- **入力表現**: Spacetime Patches — 動画を時空間パッチに分割してトークン化
- **条件付け**: adaLN-Zero + テキストエンコーダ（T5系）
- **特長**: 最大60秒の長尺動画生成、可変解像度・アスペクト比
- **Phase 6との対応**: Notebook 130（Temporal Attention）+ 132（DiT）

#### Runway Gen-2 / Gen-3

- **アーキテクチャ**: U-Net系 Video Diffusion
- **時間モデリング**: Temporal Attention層 + Temporal Conv
- **条件付け**: CLIP + Cross-Attention
- **特長**: テキスト/画像からの動画生成、スタイル制御
- **Phase 6との対応**: Notebook 130 + 131（Video Diffusion U-Net）

#### Stable Video Diffusion (Stability AI)

- **アーキテクチャ**: Stable Diffusion U-Net + Temporal層
- **時間モデリング**: 画像SDの各ブロックにtemporal attention/convを追加
- **条件付け**: 画像条件付け（Image-to-Video）
- **特長**: オープンソース、画像からの動画生成に特化
- **Phase 6との対応**: Notebook 131 + 133（事前学習活用）

#### Kling (Kuaishou)

- **アーキテクチャ**: DiT系（3D VAE + DiT）
- **特長**: 高品質な動きの表現、物理的一貫性
- **Phase 6との対応**: Notebook 132 + 135（物理整合性）

#### Pika

- **アーキテクチャ**: U-Net/DiT ハイブリッド（詳細非公開）
- **特長**: 簡便なUI、エフェクト編集機能
- **Phase 6との対応**: Notebook 131 + 134（制御性）

In [ ]:
# ============================================================
# 商用動画生成AIの技術比較表
# ============================================================

def plot_commercial_ai_comparison():
    """商用動画生成AIの技術的位置づけを表形式で可視化"""
    
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.axis('off')
    
    # テーブルデータ
    columns = ['製品', 'ベースアーキ', '時間モデリング', '最大長', '解像度', '公開時期']
    data = [
        ['Sora (OpenAI)',           'DiT',           'Spacetime Patches',   '~60s',  '1080p',  '2024.02'],
        ['Runway Gen-3',           'U-Net系',        'Temporal Attn+Conv',  '~10s',  '1080p',  '2024.06'],
        ['Stable Video Diffusion', 'SD U-Net+Temp',  'Temporal層追加',       '~4s',   '576p',   '2023.11'],
        ['Kling (Kuaishou)',       'DiT系',          '3D VAE + DiT',        '~10s',  '1080p',  '2024.06'],
        ['Pika 1.0',              'ハイブリッド',      '非公開',               '~4s',   '1080p',  '2023.12'],
        ['Veo 2 (Google)',         'DiT系',          'Spacetime Attn',      '~120s', '4K',     '2024.12'],
    ]
    
    # 色設定
    cell_colors = []
    header_color = '#4472C4'
    row_colors = ['#F2F7FB', '#FFFFFF']
    
    for i in range(len(data)):
        cell_colors.append([row_colors[i % 2]] * len(columns))
    
    table = ax.table(
        cellText=data,
        colLabels=columns,
        cellColours=cell_colors,
        colColours=[header_color] * len(columns),
        loc='center',
        cellLoc='center',
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.0, 2.0)
    
    # ヘッダーのテキスト色を白に
    for j in range(len(columns)):
        table[0, j].get_text().set_color('white')
        table[0, j].get_text().set_fontweight('bold')
    
    ax.set_title('商用動画生成AI の技術比較', fontsize=15, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

plot_commercial_ai_comparison()

In [ ]:
# ============================================================
# 商用動画生成AIのタイムライン
# ============================================================

def plot_video_ai_timeline():
    """動画生成AIの進化タイムラインを可視化"""
    
    fig, ax = plt.subplots(figsize=(18, 8))
    
    # イベントデータ: (年月, y位置, ラベル, 色, アーキテクチャ)
    events = [
        # 研究
        (2022.3,  1.5, 'Video Diffusion\nModels (Ho et al.)', 'steelblue', 'U-Net系'),
        (2022.9,  2.5, 'Make-A-Video\n(Meta)', 'steelblue', 'U-Net系'),
        (2022.10, 0.5, 'Imagen Video\n(Google)', 'steelblue', 'U-Net系'),
        (2023.3,  1.5, 'Gen-2\n(Runway)', 'coral', '商用'),
        (2023.7,  2.5, 'Pika Beta', 'coral', '商用'),
        (2023.11, 0.5, 'Stable Video\nDiffusion', 'seagreen', 'オープンソース'),
        (2023.12, 1.5, 'Pika 1.0', 'coral', '商用'),
        (2024.2,  3.0, 'Sora\n(OpenAI)', 'gold', 'DiT系'),
        (2024.5,  2.0, 'Latte / Open-Sora', 'seagreen', 'オープンソース'),
        (2024.6,  1.0, 'Gen-3 Alpha\n(Runway)', 'coral', '商用'),
        (2024.6,  3.5, 'Kling\n(Kuaishou)', 'gold', 'DiT系'),
        (2024.12, 2.5, 'Veo 2\n(Google)', 'gold', 'DiT系'),
    ]
    
    # タイムライン軸
    ax.axhline(y=0, color='gray', linewidth=2, alpha=0.5)
    
    for time, height, label, color, arch in events:
        # 縦線
        ax.plot([time, time], [0, height], color=color, linewidth=1.5, alpha=0.6)
        # ドット
        ax.plot(time, height, 'o', color=color, markersize=10, zorder=5)
        # ラベル
        ax.text(time, height + 0.15, label, ha='center', va='bottom',
                fontsize=8.5, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.15))
    
    # 年ラベル
    for year in [2022, 2023, 2024]:
        ax.text(year + 0.5, -0.4, str(year), ha='center', fontsize=14, fontweight='bold')
    
    # 凡例
    legend_elements = [
        mpatches.Patch(color='steelblue', alpha=0.6, label='研究論文 (U-Net系)'),
        mpatches.Patch(color='coral', alpha=0.6, label='商用製品'),
        mpatches.Patch(color='seagreen', alpha=0.6, label='オープンソース'),
        mpatches.Patch(color='gold', alpha=0.6, label='DiT系 (次世代)'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=11)
    
    # アノテーション: パラダイムシフト
    ax.annotate('U-Net時代 --> DiT時代へ', xy=(2024.0, 3.8),
                fontsize=12, fontweight='bold', color='darkred',
                ha='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    ax.set_xlim(2022.0, 2025.2)
    ax.set_ylim(-0.8, 4.5)
    ax.set_xlabel('年', fontsize=12)
    ax.set_title('動画生成AI の進化タイムライン (2022-2024)',
                 fontsize=15, fontweight='bold')
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

plot_video_ai_timeline()

In [ ]:
# ============================================================
# 商用AI と Phase 6 ノートブックの対応マッピング
# ============================================================

def plot_notebook_product_mapping():
    """各商用製品がPhase 6のどの技術を使っているかをマッピング"""
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # ノートブック（横軸）
    notebooks = ['NB130\nTemporal\nAttn', 'NB131\nVideo\nDiffusion', 'NB132\nDiT',
                 'NB133\n(2+1)D\nConv', 'NB134\n制御', 'NB135\n物理\n整合性']
    # 製品（縦軸）
    products = ['Sora', 'Runway Gen-3', 'SVD', 'Kling', 'Pika', 'Veo 2']
    
    # 対応マトリクス (0=無関係, 1=関連, 2=核心技術)
    mapping = np.array([
        [2, 1, 2, 1, 1, 1],  # Sora
        [2, 2, 1, 1, 2, 1],  # Runway Gen-3
        [2, 2, 0, 2, 1, 0],  # SVD
        [2, 1, 2, 1, 1, 2],  # Kling
        [1, 2, 1, 1, 2, 1],  # Pika
        [2, 1, 2, 1, 1, 1],  # Veo 2
    ])
    
    # ヒートマップ
    cmap = plt.cm.YlOrRd
    im = ax.imshow(mapping, cmap=cmap, aspect='auto', vmin=0, vmax=2)
    
    ax.set_xticks(range(len(notebooks)))
    ax.set_xticklabels(notebooks, fontsize=10)
    ax.set_yticks(range(len(products)))
    ax.set_yticklabels(products, fontsize=11, fontweight='bold')
    
    # セル内のラベル
    labels = {0: '-', 1: 'o', 2: '*'}
    for i in range(len(products)):
        for j in range(len(notebooks)):
            text_color = 'white' if mapping[i, j] >= 2 else 'black'
            ax.text(j, i, labels[mapping[i, j]], ha='center', va='center',
                    fontsize=16, fontweight='bold', color=text_color)
    
    # 凡例
    legend_text = '  * = 核心技術  |  o = 関連技術  |  - = 非該当'
    ax.text(2.5, -0.8, legend_text, ha='center', fontsize=11)
    
    ax.set_title('商用動画生成AI と Phase 6 ノートブックの対応',
                 fontsize=15, fontweight='bold', pad=15)
    
    plt.colorbar(im, ax=ax, shrink=0.6, label='関連度')
    plt.tight_layout()
    plt.show()

plot_notebook_product_mapping()

### パラダイムシフトの観察

タイムラインから読み取れる重要なトレンド：

1. **2022年: 研究フェーズ** — Video Diffusion Models, Make-A-Video, Imagen Video など
   学術研究が先行し、U-Netベースのアーキテクチャが主流

2. **2023年: 商用化フェーズ** — Runway Gen-2, Pika, Stable Video Diffusion
   研究成果が急速に商用製品に転化。U-Net系が中心

3. **2024年: DiTへの移行** — Sora, Kling, Veo 2
   Diffusion Transformer (DiT) が品質とスケーラビリティで優位性を示す

このパラダイムシフトは、画像生成で起きた「GAN → 拡散モデル」の移行と同様の構造的変化です。
Transformerのスケーリング則（Notebook 132）が、動画生成でも決定的に重要であることが
実証されました。

<a id="section4"></a>
## 4. 残された課題

動画生成AIは急速に進歩していますが、依然として根本的な課題が残されています。

### 課題1: 長尺動画生成（> 1分）

**現状**: ほとんどのモデルは4-10秒の短い動画しか生成できない（Soraの60秒は例外的）

**技術的ボトルネック**:
- メモリ: T=240フレーム(10秒@24fps) でAttentionの計算量が爆発
- 一貫性: 長いほどフレーム間の矛盾が蓄積
- 計画性: 長期的なシーンの展開を計画する能力が不足

**アプローチ**:
- Autoregressive拡張: 数秒ずつ生成して連結
- 階層的生成: キーフレーム生成 → 補間
- Sliding Window Attention: 局所的なAttentionで長尺に対応

---

### 課題2: 物理的一貫性

**現状**: 重力に反する動き、物体の消失・融合、非物理的な変形が頻発

**具体的な問題**:
- 物体の恒常性（Object Permanence）: 遮蔽されると消える
- ニュートン力学: 放物線運動、衝突応答が不正確
- 流体力学: 水や煙の挙動が非現実的

**アプローチ（Notebook 135で議論）**:
- 物理損失関数の導入
- 物理シミュレータとの統合
- 物理エンジンベースの訓練データ拡充

---

### 課題3: 制御性（Controllability）

**現状**: テキストプロンプトだけでは細かいカメラワークや物体運動を指定できない

**必要な制御軸（Notebook 134で議論）**:
- カメラ制御: 6DoF（位置3 + 回転3）の精密制御
- 物体運動: 個別物体のトラジェクトリ指定
- シーン編集: 一部のみの変更（背景維持で物体だけ動かす等）

**アプローチ**:
- ControlNet系の制御信号注入
- 深度マップ・光学フロー条件付け
- インタラクティブな編集インターフェース

---

### 課題4: リアルタイム生成

**現状**: 数秒の動画生成に数分かかる（拡散の反復推論がボトルネック）

**目標**: インタラクティブなアプリケーション（ゲーム、VR等）に必要な < 100ms/フレーム

**アプローチ**:
- 蒸留（Distillation）: ステップ数を50 → 4に削減
- Consistency Models: 1ステップ生成
- ハードウェア最適化: 専用チップ、量子化

### 課題の整理と期待されるブレークスルー

| 課題 | 現状の限界 | 期待されるブレークスルー | 関連技術 |
|------|-----------|---------------------|----------|
| 長尺動画 | ~10秒が実用的限界 | 階層的生成 + Sliding Window | Temporal Attention拡張 |
| 物理整合性 | 重力・衝突が非現実的 | 物理エンジンとの統合 | 世界モデル (Phase 7) |
| カメラ制御 | テキストでは精密制御困難 | 6DoF直接指定 | ControlNet動画版 |
| 物体運動 | 個別物体の精密操作が困難 | トラジェクトリ条件付け | Optical Flow制御 |
| リアルタイム | 数分/数秒の動画 | 蒸留 + Consistency Model | 推論効率化 |
| 時間的一貫性 | 長いと矛盾が蓄積 | 潜在空間での状態追跡 | JEPA (Phase 7) |

これらの課題の多くは、Phase 7「世界モデル」で取り組む技術によって
根本的な解決に近づくと期待されています。

In [ ]:
# ============================================================
# 残された課題の構造化マップ
# ============================================================

def plot_remaining_challenges():
    """残された課題を難易度・重要度のマトリクスで可視化"""
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 課題: (重要度, 難易度, ラベル, 色)
    challenges = [
        (4.5, 4.8, '長尺動画生成\n(>1分)', 'coral', 14),
        (4.8, 4.5, '物理的一貫性', 'red', 14),
        (4.0, 3.5, 'カメラ制御', 'steelblue', 12),
        (3.5, 3.0, '物体運動制御', 'steelblue', 12),
        (3.0, 4.0, 'リアルタイム\n生成', 'seagreen', 13),
        (3.8, 3.8, 'テキスト-動画\n整合性', 'orange', 12),
        (2.5, 2.5, '解像度向上\n(4K/8K)', 'gray', 11),
        (4.2, 3.2, '時間的一貫性', 'purple', 12),
    ]
    
    for importance, difficulty, label, color, fontsize in challenges:
        ax.scatter(difficulty, importance, s=400, c=color, alpha=0.6, edgecolors='gray', zorder=5)
        ax.annotate(label, (difficulty, importance),
                    textcoords='offset points', xytext=(15, 5),
                    fontsize=fontsize, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.1))
    
    # 象限ラベル
    ax.text(1.8, 4.5, '高重要・低難度\n(優先的に取り組む)', fontsize=10,
            ha='center', va='center', color='green', alpha=0.5, style='italic')
    ax.text(4.5, 4.5, '高重要・高難度\n(長期的研究課題)', fontsize=10,
            ha='center', va='center', color='red', alpha=0.5, style='italic')
    ax.text(1.8, 1.5, '低重要・低難度\n(付加的な改善)', fontsize=10,
            ha='center', va='center', color='gray', alpha=0.5, style='italic')
    ax.text(4.5, 1.5, '低重要・高難度\n(後回し可)', fontsize=10,
            ha='center', va='center', color='gray', alpha=0.5, style='italic')
    
    # 中央の区切り線
    ax.axhline(y=3.0, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(x=3.0, color='gray', linestyle='--', alpha=0.3)
    
    ax.set_xlabel('技術的難易度', fontsize=13)
    ax.set_ylabel('実用上の重要度', fontsize=13)
    ax.set_title('動画生成AI の残された課題マップ',
                 fontsize=15, fontweight='bold')
    ax.set_xlim(1.2, 5.3)
    ax.set_ylim(1.2, 5.3)
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

plot_remaining_challenges()

<a id="section5"></a>
## 5. Phase 7へのロードマップ

### 「動画を生成する」から「世界を理解する」へ

Phase 6では「動画を生成する」技術を学びました。
Phase 7では、その先にある**「世界を理解し、予測し、インタラクションする」**技術に進みます。

この遷移は、AIの歴史における根本的なパラダイムシフトです：

```
Phase 6: 動画生成            Phase 7: 世界モデル
                      
テキスト/画像 → 動画      世界の状態 → 予測 → 行動
「見た目がリアル」        「物理法則を理解」
「ピクセル空間で学習」    「潜在空間で世界をモデル化」
「受動的な生成」          「能動的な予測と制御」
```

### なぜ世界モデルが必要か？

動画生成モデルの限界が世界モデルの必要性を示しています：

1. **物理整合性の根本的解決**: ピクセル空間での損失関数では物理法則を完全に捕捉できない
   → 世界の物理構造を内部表現として学習する必要がある

2. **長期予測**: 数秒以上の動画生成で一貫性が崩れる
   → 高レベルの状態遷移を学習すれば長期計画が可能

3. **インタラクション**: 動画生成は「見る」だけ。ロボティクスやゲームでは「行動」が必要
   → 行動の結果を予測する世界モデルが不可欠

### Phase 7で学ぶ技術

| 技術 | 概要 | Phase 6からの接続 |
|------|------|------------------|
| **JEPA** (Joint Embedding Predictive Architecture) | 潜在空間で未来を予測（LeCunの提案） | 動画の潜在表現学習の発展 |
| **Dreamer** | 夢の中で学習する強化学習エージェント | 動画予測 → 行動計画への拡張 |
| **Genie** (Google DeepMind) | 動画からインタラクティブな世界を生成 | 動画生成 + ユーザー行動の統合 |
| **Model-based RL** | 世界モデルを使った効率的な強化学習 | 動画予測の応用先 |

In [ ]:
# ============================================================
# Phase 6 → Phase 7 の遷移図
# ============================================================

def plot_phase_transition():
    """Phase 6からPhase 7への技術的遷移を可視化"""
    
    fig, ax = plt.subplots(figsize=(18, 10))
    
    # Phase 6 ブロック
    phase6_items = [
        ('Temporal\nAttention', 0.08, 0.75, 'lightblue'),
        ('Video\nDiffusion', 0.08, 0.55, 'lightgreen'),
        ('DiT', 0.08, 0.35, 'lightyellow'),
    ]
    
    for label, x, y, color in phase6_items:
        rect = mpatches.FancyBboxPatch(
            (x, y), 0.13, 0.12,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='gray', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(x + 0.065, y + 0.06, label, ha='center', va='center',
                fontsize=10, fontweight='bold')
    
    # Phase 6 ラベル
    rect_p6 = mpatches.FancyBboxPatch(
        (0.03, 0.28), 0.23, 0.65,
        boxstyle='round,pad=0.02',
        facecolor='none', edgecolor='steelblue', linewidth=3, linestyle='--'
    )
    ax.add_patch(rect_p6)
    ax.text(0.145, 0.92, 'Phase 6:\n動画生成', ha='center', va='center',
            fontsize=13, fontweight='bold', color='steelblue')
    
    # 橋渡し（課題）
    bridge_items = [
        ('物理整合性の限界', 0.35, 0.72),
        ('長期予測の困難', 0.35, 0.55),
        ('行動との統合が不可', 0.35, 0.38),
    ]
    
    for label, x, y in bridge_items:
        ax.text(x, y, label, ha='center', va='center',
                fontsize=10, color='darkred', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                          edgecolor='orange', alpha=0.8))
        # 矢印
        ax.annotate('', xy=(x - 0.08, y), xytext=(0.21, y),
                    arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
        ax.annotate('', xy=(0.50, y), xytext=(x + 0.08, y),
                    arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
    
    # Phase 7 ブロック
    phase7_items = [
        ('JEPA\n潜在空間予測', 0.55, 0.75, '#D4EDDA'),
        ('Dreamer\n夢の中で学習', 0.55, 0.55, '#D4EDDA'),
        ('Genie\n世界との対話', 0.55, 0.35, '#D4EDDA'),
        ('Model-based RL\n効率的な計画', 0.75, 0.55, '#D4EDDA'),
    ]
    
    for label, x, y, color in phase7_items:
        rect = mpatches.FancyBboxPatch(
            (x, y), 0.15, 0.12,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='gray', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(x + 0.075, y + 0.06, label, ha='center', va='center',
                fontsize=10, fontweight='bold')
    
    # Phase 7 内部接続
    ax.annotate('', xy=(0.75, 0.65), xytext=(0.66, 0.75),
                arrowprops=dict(arrowstyle='->', color='green', lw=1.2, alpha=0.5))
    ax.annotate('', xy=(0.75, 0.58), xytext=(0.70, 0.58),
                arrowprops=dict(arrowstyle='->', color='green', lw=1.2, alpha=0.5))
    ax.annotate('', xy=(0.75, 0.50), xytext=(0.66, 0.40),
                arrowprops=dict(arrowstyle='->', color='green', lw=1.2, alpha=0.5))
    
    # Phase 7 ラベル
    rect_p7 = mpatches.FancyBboxPatch(
        (0.50, 0.28), 0.45, 0.65,
        boxstyle='round,pad=0.02',
        facecolor='none', edgecolor='green', linewidth=3, linestyle='--'
    )
    ax.add_patch(rect_p7)
    ax.text(0.725, 0.92, 'Phase 7:\n世界モデル', ha='center', va='center',
            fontsize=13, fontweight='bold', color='green')
    
    # 全体タイトル
    ax.text(0.5, 0.15, '「動画を生成する」から「世界を理解し、予測し、行動する」へ',
            ha='center', va='center', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='gold', alpha=0.3))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0.05, 1.0)
    ax.axis('off')
    ax.set_title('Phase 6 から Phase 7 への遷移',
                 fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

plot_phase_transition()

In [ ]:
# ============================================================
# 抽象化レベルの比較: 動画生成 vs 世界モデル
# ============================================================

def plot_abstraction_levels():
    """動画生成と世界モデルの抽象化レベルを比較"""
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # 階層構造
    levels = [
        (0.15, 0.15, 'ピクセル空間\n(Raw Pixels)', 'lightblue', 0.65, 0.10),
        (0.15, 0.35, 'パッチ / トークン空間\n(Spacetime Patches)', 'lightgreen', 0.55, 0.10),
        (0.15, 0.55, '潜在表現空間\n(Latent Space)', '#D4EDDA', 0.45, 0.10),
        (0.15, 0.75, '状態空間\n(World State)', 'gold', 0.35, 0.10),
        (0.15, 0.90, '行動・計画空間\n(Action/Plan)', 'lightsalmon', 0.25, 0.08),
    ]
    
    for x, y, label, color, w, h in levels:
        rect = mpatches.FancyBboxPatch(
            (x, y - h/2), w, h,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='gray', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(x + w/2, y, label, ha='center', va='center',
                fontsize=10, fontweight='bold')
    
    # Phase 6 の範囲
    rect_p6 = mpatches.FancyBboxPatch(
        (0.82, 0.08), 0.12, 0.42,
        boxstyle='round,pad=0.01',
        facecolor='none', edgecolor='steelblue', linewidth=3, linestyle='--'
    )
    ax.add_patch(rect_p6)
    ax.text(0.88, 0.30, 'Phase 6\n動画生成', ha='center', va='center',
            fontsize=11, fontweight='bold', color='steelblue')
    
    # Phase 7 の範囲
    rect_p7 = mpatches.FancyBboxPatch(
        (0.82, 0.50), 0.12, 0.45,
        boxstyle='round,pad=0.01',
        facecolor='none', edgecolor='green', linewidth=3, linestyle='--'
    )
    ax.add_patch(rect_p7)
    ax.text(0.88, 0.72, 'Phase 7\n世界モデル', ha='center', va='center',
            fontsize=11, fontweight='bold', color='green')
    
    # 抽象度ラベル
    ax.annotate('', xy=(0.05, 0.90), xytext=(0.05, 0.15),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax.text(0.02, 0.52, '抽象度', fontsize=10, rotation=90,
            ha='center', va='center', color='gray', fontweight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('抽象化レベルの階層: Phase 6 と Phase 7 の守備範囲',
                 fontsize=15, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('Phase 6はピクセル~潜在空間で「動画を生成する」技術を扱いました。')
    print('Phase 7では潜在空間~行動空間で「世界を理解し行動する」技術に進みます。')

plot_abstraction_levels()

### Phase 7 プレビュー: 各技術の概要

#### JEPA (Joint Embedding Predictive Architecture)

Yann LeCun が提唱するアーキテクチャで、**ピクセル空間ではなく潜在空間で未来を予測**します。

- 動画のフレーム列を潜在表現にエンコード
- 潜在空間で次のフレームの表現を予測（ピクセルを生成しない）
- GAN/拡散モデルの「ピクセル空間での学習」の限界を克服

Phase 6との接続: DiT（Notebook 132）のエンコーダ部分を、予測タスクに転用するイメージ

---

#### Dreamer

強化学習エージェントが**「夢の中」（学習した世界モデル内）で計画を練る**手法です。

- 環境を観察して内部世界モデル（動画予測モデルに相当）を構築
- 世界モデル内でシミュレーションを行い、最適な行動計画を探索
- 実世界での試行錯誤を大幅に削減

Phase 6との接続: 動画拡散モデル（Notebook 131）が予測する「未来のフレーム」を行動選択に活用

---

#### Genie (Google DeepMind)

**動画からインタラクティブな仮想世界を生成**する技術です。

- ユーザーの行動（キー入力等）を条件として次のフレームを生成
- 2Dゲームのような世界を動画から自動的に構築
- 学習データは「ラベルなし動画」のみ（行動のアノテーション不要）

Phase 6との接続: 条件付き動画生成（Notebook 134）の制御軸を「ユーザー行動」に拡張

In [ ]:
# ============================================================
# Phase 6 → Phase 7 の技術的接続を整理
# ============================================================

def print_phase7_roadmap():
    """Phase 7ロードマップの詳細"""
    print("=" * 70)
    print("Phase 7 ロードマップ: 世界モデルへの道")
    print("=" * 70)
    
    roadmap = [
        ("JEPA",
         "Joint Embedding Predictive Architecture",
         "潜在空間で未来を予測する",
         "Phase 6で学んだ動画の潜在表現(VAE/DiT encoder)を\n"
         "     予測タスクに転用。ピクセル生成からの脱却。"),
        ("Dreamer",
         "Model-based RL with World Models",
         "夢の中(世界モデル内)で行動を計画する",
         "Phase 6の動画予測モデルを『夢を見るエンジン』として使い、\n"
         "     強化学習エージェントの行動計画を効率化。"),
        ("Genie",
         "Generative Interactive Environments",
         "動画からインタラクティブな世界を生成する",
         "Phase 6の条件付き動画生成(Notebook 134)の制御軸を\n"
         "     ユーザー行動に拡張し、仮想世界を自動構築。"),
        ("Model-based RL",
         "Sample-efficient Reinforcement Learning",
         "世界モデルでシミュレーションして効率的に学習する",
         "物理整合性(Notebook 135)をさらに発展させ、\n"
         "     物理法則を内在化した世界モデルで行動を最適化。"),
    ]
    
    for name, full_name, summary, connection in roadmap:
        print(f"\n  --- {name} ({full_name}) ---")
        print(f"  概要: {summary}")
        print(f"  接続: {connection}")
    
    print("\n" + "=" * 70)
    print("Phase 7のキーメッセージ:")
    print("  動画生成は『世界を理解するAI』への重要な中間ステップです。")
    print("  Phase 6の技術基盤の上に、世界モデルという新しい層が構築されます。")

print_phase7_roadmap()

<a id="quiz"></a>
## 6. 総合クイズ

Phase 6全体の理解を確認する5問のクイズです。
各問題は異なるノートブックの内容をカバーしています。

---

### Q1: Temporal Attention と Spatial Attention (Notebook 130)

**問題**: 動画テンソル `(B, T, C, H, W)` に対して Temporal Self-Attention を適用するとき、
テンソルをどのようにreshapeしてからAttention計算を行いますか？
また、Spatial Self-Attention の場合はどうreshapeしますか？
その違いが生じる理由を説明してください。

<details>
<summary>答えを見る</summary>

**答え**:

- **Temporal Attention**: `(B*H*W, T, C)` にreshape
  - 各空間位置をバッチ次元に畳み込み、時間軸T方向でAttentionを計算
  - 同じ空間位置の異なるフレーム間の関係を捉える

- **Spatial Attention**: `(B*T, H*W, C)` にreshape
  - 各フレームをバッチ次元に畳み込み、空間軸H*W方向でAttentionを計算
  - 同じフレーム内の異なる空間位置間の関係を捉える

**理由**: Attentionの数式自体は同一（QKV計算 + softmax + 加重和）です。
reshapeによって「何をトークン列として扱うか」を切り替えるだけで、
Spatial/Temporalの役割が変わります。これが「reshapeだけで切り替え可能」という
Notebook 130の核心的洞察です。

</details>

---

### Q2: (2+1)D Convolution の利点 (Notebook 131, 133)

**問題**: 3D Convolution を (2+1)D Convolution（空間2D + 時間1D の分離）に
置き換えることの利点を3つ挙げてください。
また、計算量はどのように変化しますか？

<details>
<summary>答えを見る</summary>

**答え**:

**3つの利点**:

1. **パラメータ効率**: 3D Conv の `C_in * C_out * k * k * k` パラメータが、
   2D部分 `C_in * C_mid * k * k` + 1D部分 `C_mid * C_out * k` に分離され、
   総パラメータ数が削減される

2. **事前学習の活用**: 空間2D Conv部分は画像モデル（Stable Diffusion等）の
   事前学習済み重みをそのまま再利用でき、時間1D Conv部分のみ新規学習すればよい

3. **学習の安定性**: 空間と時間を別々に学習することで、
   最適化が容易になり訓練が安定する（非線形性の追加により表現力も向上）

**計算量**: 3D Conv: O(C_in * C_out * k^3 * T * H * W) に対し、
(2+1)D Conv は O(C_in * C_mid * k^2 * T * H * W + C_mid * C_out * k * T * H * W)
となり、k=3の場合で約30-50%の削減が可能

</details>

---

### Q3: adaLN-Zero の仕組み (Notebook 132)

**問題**: DiT (Diffusion Transformer) で採用されている adaLN-Zero とは何ですか？
通常の条件付け手法（Cross-Attention等）と比較した利点を説明してください。

<details>
<summary>答えを見る</summary>

**答え**:

**adaLN-Zero (Adaptive Layer Normalization with Zero-initialization)**:

- LayerNormのスケール(gamma)とシフト(beta)パラメータを、
  条件情報（拡散タイムステップ、クラスラベル等）から動的に生成する手法
- `output = gamma(condition) * LayerNorm(x) + beta(condition)`
- さらに残差接続の前にもう一つのスケーリングパラメータ(alpha)を追加
- 初期化時に alpha=0 とすることで、訓練初期は恒等写像として振る舞う（Zero-init）

**Cross-Attentionとの比較**:

| 項目 | adaLN-Zero | Cross-Attention |
|------|-----------|----------------|
| パラメータ数 | 少ない（線形層のみ） | 多い（Q,K,V投影） |
| 計算コスト | 低い | 高い |
| 条件の種類 | スカラー/ベクトル（timestep等） | シーケンス（テキスト等） |
| 訓練安定性 | 高い（Zero-init） | 中程度 |

adaLN-Zeroは「グローバルな条件」（拡散ステップ、クラス）に特に有効で、
テキスト条件のような「シーケンス条件」にはCross-Attentionが適しています。

</details>

---

### Q4: カメラ運動と物体運動の制御 (Notebook 134)

**問題**: 動画生成における「カメラ運動の制御」と「物体運動の制御」の技術的な違いを
説明してください。それぞれどのような条件情報を使いますか？

<details>
<summary>答えを見る</summary>

**答え**:

**カメラ運動の制御**:
- **条件情報**: カメラの外部パラメータ（位置 [x,y,z] + 回転 [pitch,yaw,roll]）の時系列
- **技術**: 6DoFのカメラポーズをベクトルとしてエンコードし、条件付け層に注入
- **特徴**: シーン全体に影響（全ピクセルが同じカメラ変換を受ける）
- **表現**: パン、ティルト、ズーム、ドリー等のカメラワーク

**物体運動の制御**:
- **条件情報**: 個別物体のトラジェクトリ（軌跡 [x,y,t]）、バウンディングボックス列、
  またはオプティカルフロー
- **技術**: 物体ごとの制御信号を空間的に注入（ControlNet拡張、深度マップ等）
- **特徴**: 局所的な影響（対象物体の領域のみ変化）
- **表現**: 物体の移動、回転、スケール変化

**根本的な違い**: カメラ運動は「視点の変化」（グローバル変換）、
物体運動は「被写体の変化」（ローカル変換）です。
両者を分離して制御できることが、実用的な動画生成の鍵です。

</details>

---

### Q5: 物理整合性の評価 (Notebook 135)

**問題**: 生成された動画の「物理的な正しさ」をどのように評価しますか？
FVD (Frechet Video Distance) だけでは不十分な理由と、
物理整合性を向上させるための手法を1つ説明してください。

<details>
<summary>答えを見る</summary>

**答え**:

**FVDが不十分な理由**:
FVDは「生成動画の分布」と「実動画の分布」の統計的距離を測る指標です。
見た目の品質や多様性は捉えますが、**個々のフレームの物理法則違反**は検出できません。

例えば、FVDが低い（良い）スコアでも：
- ボールが重力に逆らって浮遊する
- 物体が壁をすり抜ける
- 水が上に向かって流れる

これらの非物理的な動画が高スコアを得る可能性があります。

**物理整合性を向上させる手法の例 — 物理損失関数**:

生成動画から光学フローを推定し、物理法則に基づく制約を損失関数に追加します。

- **運動保存損失**: 遮蔽がない領域での物体の速度変化が滑らかであることを制約
- **重力損失**: 落下する物体の加速度がg (9.8m/s^2) に近いことを制約
- **衝突損失**: 物体が重なる（貫通する）ことがないように制約

これらを通常の拡散損失（MSE）に加えて最適化することで、
物理的により妥当な動画を生成できるようになります。

</details>

In [ ]:
# ============================================================
# クイズの対応ノートブック一覧
# ============================================================

def print_quiz_mapping():
    """各クイズ問題とノートブックの対応を表示"""
    print("=" * 60)
    print("総合クイズ — ノートブック対応表")
    print("=" * 60)
    
    mapping = [
        ("Q1", "Temporal vs Spatial Attention", "130",
         "reshapeの仕組みと直感的理解"),
        ("Q2", "(2+1)D Conv の利点", "131, 133",
         "分離畳み込みの効率性と事前学習活用"),
        ("Q3", "adaLN-Zero の仕組み", "132",
         "DiTにおける条件付けの設計思想"),
        ("Q4", "カメラ運動 vs 物体運動", "134",
         "グローバル vs ローカルな制御の違い"),
        ("Q5", "物理整合性の評価", "135",
         "FVDの限界と物理損失関数"),
    ]
    
    for qid, topic, notebooks, focus in mapping:
        print(f"\n  {qid}: {topic}")
        print(f"    対応: Notebook {notebooks}")
        print(f"    焦点: {focus}")
    
    print("\n" + "=" * 60)
    print("全問正解できれば、Phase 6の技術体系を十分に理解しています。")
    print("不安な問題があれば、対応するノートブックを復習してください。")

print_quiz_mapping()

### 自習のための参考文献と推奨リソース

Phase 6で学んだ内容をさらに深めたい場合の参考文献を整理します。

#### 論文

| 論文 | 対応NB | 内容 |
|------|--------|------|
| Ho et al., "Video Diffusion Models" (2022) | 130, 131 | 動画拡散モデルの基礎論文 |
| Peebles & Xie, "Scalable Diffusion Models with Transformers" (2023) | 132 | DiTの原著論文 |
| Tran et al., "A Closer Look at Spatiotemporal Convolutions" (2018) | 133 | (2+1)D Convの原著 |
| Blattmann et al., "Stable Video Diffusion" (2023) | 131, 133 | SVDの技術報告 |
| Unterthiner et al., "FVD: A new Metric for Video Generation" (2019) | 135 | FVDの原著論文 |

#### Phase 7 準備

| 論文/リソース | 対応Phase 7テーマ |
|-------------|------------------|
| LeCun, "A Path Towards Autonomous Machine Intelligence" (2022) | JEPA |
| Hafner et al., "Dream to Control" (DreamerV3, 2023) | Dreamer |
| Bruce et al., "Genie: Generative Interactive Environments" (2024) | Genie |
| Sutton, "The Bitter Lesson" (2019) | スケーリング哲学 |

<a id="summary"></a>
## 7. まとめとチェックリスト

### Phase 6 学習チェックリスト

以下の全ての項目にチェックが入れば、Phase 6の学習は完了です。

#### 基礎理解（Notebook 130）
- [ ] Spatial / Temporal / Spatiotemporal Attentionの違いを説明できる
- [ ] `(B, T, C, H, W)` テンソルを各Attention用にreshapeできる
- [ ] 因果マスクの仕組みと必要性を理解している

#### 実装力（Notebook 131）
- [ ] U-Netにtemporal層を追加して動画拡散モデルを構築できる
- [ ] (2+1)D Convolutionを実装できる
- [ ] 動画拡散モデルの訓練パイプラインを構築できる

#### 発展的理解（Notebook 132）
- [ ] DiTのアーキテクチャとadaLN-Zeroを説明できる
- [ ] Spacetime Patchesの概念を理解している
- [ ] U-Net vs DiTのトレードオフを議論できる

#### 効率化（Notebook 133）
- [ ] 3D Conv vs (2+1)D Convの計算量を比較できる
- [ ] 事前学習重みの転移方法を理解している

#### 制御性（Notebook 134）
- [ ] カメラ運動と物体運動の制御方法の違いを説明できる
- [ ] ControlNet拡張の動画版を理解している

#### 品質保証（Notebook 135）
- [ ] FVDの意味と限界を理解している
- [ ] 物理損失関数による品質向上手法を説明できる
- [ ] 時間的一貫性の評価方法を理解している

#### 総合力（本章）
- [ ] 3つのアーキテクチャの特徴を比較できる
- [ ] 商用動画生成AIの技術的位置づけを説明できる
- [ ] Phase 7（世界モデル）への接続を理解している

In [ ]:
# ============================================================
# 最終サマリとPhase 7への誘導
# ============================================================

def final_summary():
    """Phase 6の最終まとめメッセージ"""
    print("=" * 70)
    print("Phase 6 完了: 時空間モデリング")
    print("=" * 70)
    
    print("""
Phase 6「時空間モデリング」の全6章 + 総括を完了しました。

【習得した技術スタック】

  Notebook 130: Temporal Attention / Causal Mask / SpatioTemporalBlock
  Notebook 131: Video Diffusion U-Net / (2+1)D Conv / Moving-MNIST
  Notebook 132: DiT / adaLN-Zero / Spacetime Patches / スケーリング則
  Notebook 133: Factorized Conv / 計算効率 / 事前学習の転移
  Notebook 134: カメラ制御 / 物体運動制御 / ControlNet拡張
  Notebook 135: 物理損失 / FVD / 時間的一貫性指標
  Notebook 136: 技術体系の統合 / 商用AIの位置づけ / Phase 7展望

【Phase 6の核心メッセージ】

  動画生成 = 画像生成 + 時間軸のモデリング
  
  この単純な拡張が、実際には以下の深い技術的課題を含みます:
  - 時間的一貫性の維持
  - 計算効率のスケーリング
  - 物理法則への準拠
  - 精密な制御性の実現

【次のステップ: Phase 7】

  Phase 7「世界モデル」では、動画生成を超えて：
  
  - JEPA: 潜在空間での予測
  - Dreamer: 世界モデルベースの強化学習
  - Genie: インタラクティブな世界生成
  - Model-based RL: 効率的な行動計画
  
  「世界を理解し、予測し、行動する」AI へと進みます。
""")
    
    print("=" * 70)
    print("Phase 6の学習お疲れ様でした。Phase 7でお会いしましょう。")
    print("=" * 70)

final_summary()